In [1]:
import pandas as pd
import requests


/Users/celine/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def get_hawker_centres(dataset_id="d_4a086da0a5553be1d89383cd90d07ecd"):
    
    poll = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    j = requests.get(poll, timeout=30).json()
    if j.get("code", 1) != 0:
        raise RuntimeError(f"data.gov.sg error: {j.get('errMsg')}")

    dl_url = j["data"]["url"]
    geojson = requests.get(dl_url, timeout=30).json()

    rows = []
    for f in geojson.get("features", []):
        p = f.get("properties", {})
        g = f.get("geometry", {})
        coords = g.get("coordinates", [None, None])  # [lon, lat]
        rows.append({
            "hawker_name": p.get("NAME"),
            "address_street": p.get("ADDRESSSTREETNAME"),
            "postal_code": p.get("ADDRESSPOSTALCODE"),
            "lat": coords[1],
            "lon": coords[0]
        })

    return pd.DataFrame(rows).drop_duplicates()


In [3]:
hawker_df = get_hawker_centres()
hawker_df.head()


,hawker_name,address_street,postal_code,lat,lon
0,Amoy Street Food Centre (Telok Ayer Food Centre),MAXWELL ROAD,069111,1.279231,103.846619
1,Margaret Drive Hawker Centre,Margaret Drive,142038,1.297487,103.804694
2,Punggol Coast Hawker Centre,Punggol Way,829911,1.414518,103.908543
3,Telok Blangah Hawker Centre & Market,Telok Blangah Drive,103078,1.273250,103.808429
4,Bukit Timah Market,Upper Bukit Timah Road,588215,1.339237,103.776330


In [4]:
hawker_df.to_csv("../../data/cleaned/hawker_centres_cleaned.csv", index=False)